In [7]:
pip install PyWavelets

   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ------------ --------------------------- 1.3/4.2 MB 8.4 MB/s eta 0:00:01
   ---------------------------------------  4.2/4.2 MB 11.4 MB/s eta 0:00:01
   ---------------------------------------- 4.2/4.2 MB 11.0 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import os

# 1. Load Metadata
meta_path = r'C:\TBC SCREENING PENELITIAN\dataset_respiscan_local\Metadata.csv'
df = pd.read_csv(meta_path)

# 2. Definisikan folder utama
base_dirs = {
    "Nairobi_New_Forced": r'C:\TBC SCREENING PENELITIAN\data_baru_8000\TBscreen_Dataset\Forced_coughs\Audio_files',
    "Nairobi_New_Passive": r'C:\TBC SCREENING PENELITIAN\data_baru_8000\TBscreen_Dataset\Passive_coughs\Audio_files',
    "Nairobi_Old": r'C:\TBC SCREENING PENELITIAN\dataset_tbc'
}

# 3. Cek distribusi label asli
print("Distribusi Label:")
print(df['Label'].value_counts()) # Pastikan 'Label' berisi 'TB' atau 'Healthy'

# 4. Fungsi untuk mencari path lengkap file berdasarkan subject ID
def find_file_path(subject_id):
    # Misal nama file mengandung subject ID, contoh: PID_73C_...
    for category, directory in base_dirs.items():
        if os.path.exists(directory):
            for filename in os.listdir(directory):
                if str(subject_id) in filename:
                    return os.path.join(directory, filename)
    return None

df['full_path'] = df['subject'].apply(find_file_path)
print(f"\nBerhasil memetakan {df['full_path'].notnull().sum()} file audio.")

Distribusi Label:
Label
TB     103
NTB     46
Name: count, dtype: int64

Berhasil memetakan 120 file audio.


In [2]:
all_records = []

print("🔎 Mencari semua rekaman untuk 120 subjek...")
for index, row in df[df['full_path'].notnull()].iterrows():
    subject_id = str(row['subject'])
    label = row['Label']
    
    # Cari di setiap base_dir
    for category, directory in base_dirs.items():
        if os.path.exists(directory):
            for filename in os.listdir(directory):
                # Jika nama file mengandung ID subjek (misal: PID_73C)
                if subject_id in filename and filename.endswith('.wav'):
                    all_records.append({
                        'subject': subject_id,
                        'Label': label,
                        'full_path': os.path.join(directory, filename),
                        'filename': filename
                    })

df_all_files = pd.DataFrame(all_records)
print(f"✅ Berhasil mengumpulkan {len(df_all_files)} total rekaman.")

🔎 Mencari semua rekaman untuk 120 subjek...
✅ Berhasil mengumpulkan 15814 total rekaman.


In [3]:
import pywt
import os
import gc
import torch
import torchaudio
import matplotlib
matplotlib.use('Agg') # Wajib agar tidak muncul jendela popup
import matplotlib.pyplot as plt

# Solusi konflik library
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def is_already_processed(filename, label, output_dir, suffix):
    # Cek apakah file gambar dan fitur MFCC sudah ada di folder
    img_path = os.path.join(output_dir, str(label), f"{filename}{suffix}.png")
    mfcc_path = os.path.join(output_dir, f"{filename}{suffix}_mfcc.pt")
    return os.path.exists(img_path) and os.path.exists(mfcc_path)

In [4]:
import os
import torch
import torchaudio
import numpy as np
import matplotlib
matplotlib.use('Agg') # Biar nggak crash di Windows
import matplotlib.pyplot as plt
import random
import gc

# --- KONFIGURASI PATH ---
output_dir = r'C:\TBC SCREENING PENELITIAN\RespiScan_Final_Dataset'
noise_dirs = [
    r'C:\TBC SCREENING PENELITIAN\DKITCHEN',
    r'C:\TBC SCREENING PENELITIAN\DLIVING',
    r'C:\TBC SCREENING PENELITIAN\NPARK',
    r'C:\TBC SCREENING PENELITIAN\OHALLWAY',
    r'C:\TBC SCREENING PENELITIAN\OOFFICE',
    r'C:\TBC SCREENING PENELITIAN\STRAFFIC'
]

# --- KONFIGURASI PERANGKAT ---
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Buat folder jika belum ada
os.makedirs(output_dir, exist_ok=True)

print(f"✅ Variabel output_dir siap: {output_dir}")
print(f"🚀 Perangkat tempur: {device}")

✅ Variabel output_dir siap: C:\TBC SCREENING PENELITIAN\RespiScan_Final_Dataset
🚀 Perangkat tempur: cuda


In [11]:
def extract_features_stable(waveform, filename, label, suffix, output_dir, device):
    # 1. MFCC (GPU)
    waveform_gpu = waveform.to(device)
    mfcc_tf = torchaudio.transforms.MFCC(sample_rate=44100, n_mfcc=13).to(device)
    mfcc = mfcc_tf(waveform_gpu)
    torch.save(mfcc.cpu(), os.path.join(output_dir, f"{filename}{suffix}_mfcc.pt"))
    
    # 2. Scalogram (CPU)
    y = waveform.numpy().flatten()
    widths = np.geomspace(1, 1024, num=100)
    cwtmatr, _ = pywt.cwt(y, widths, 'cmor1.5-1.0')
    
    # Simpan Gambar dengan efisiensi tinggi
    lbl_folder = os.path.join(output_dir, str(label))
    os.makedirs(lbl_folder, exist_ok=True)
    
    fig = plt.figure(figsize=(4.48, 2.24))
    plt.imshow(np.abs(cwtmatr), aspect='auto', cmap='jet')
    plt.axis('off')
    plt.savefig(os.path.join(lbl_folder, f"{filename}{suffix}.png"), bbox_inches='tight', pad_inches=0, dpi=100)
    
    # PEMBERSIHAN TOTAL (Mencegah Memory Leak)
    plt.cla()
    plt.clf()
    plt.close(fig)
    del waveform_gpu, mfcc, cwtmatr, y

In [12]:
def process_mfcc_gpu(waveform, filename, output_dir, device):
    # --- EKSTRAKSI MFCC via CUDA ---
    waveform_gpu = waveform.to(device)
    mfcc_tf = T.MFCC(sample_rate=44100, n_mfcc=13).to(device)
    mfcc = mfcc_tf(waveform_gpu)
    
    # Simpan tensor MFCC (Cepat)
    torch.save(mfcc.cpu(), os.path.join(output_dir, f"{filename}_mfcc.pt"))

In [14]:
import torch
import torchaudio
import torchaudio.transforms as T
import os
import random

def standardize_audio(waveform, sr, target_sr=44100, target_samples=44100):
    # 1. Pastikan Mono (1 Channel)
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)
    
    # 2. Resampling ke 44.1 kHz
    if sr != target_sr:
        resampler = T.Resample(sr, target_sr)
        waveform = resampler(waveform)
    
    # 3. Paksa Panjang tepat 1 detik (44100 sampel)
    if waveform.shape[1] > target_samples:
        waveform = waveform[:, :target_samples] # Potong jika kepanjangan
    else:
        padding = target_samples - waveform.shape[1]
        waveform = torch.nn.functional.pad(waveform, (0, padding)) # Tambah diam jika kependekan
        
    return waveform

def get_random_noise_segment(dirs):
    selected_dir = random.choice(dirs)
    files = [f for f in os.listdir(selected_dir) if f.endswith('.wav')]
    path = os.path.join(selected_dir, random.choice(files))
    
    # Load potongan acak (misal ambil 2 detik dulu untuk diproses)
    info = torchaudio.info(path)
    start_frame = random.randint(0, max(0, info.num_frames - 88200))
    noise_wf, sr = torchaudio.load(path, frame_offset=start_frame, num_frames=88200)
    
    # Gunakan fungsi standardize_audio agar ukurannya pas 1 detik
    return standardize_audio(noise_wf, sr)

def mix_noise(clean_wf, noise_wf, snr_db=10):
    # Rumus SNR untuk kekuatan sinyal
    s_pwr = clean_wf.pow(2).mean()
    n_pwr = noise_wf.pow(2).mean()
    snr = 10 ** (snr_db / 10)
    scale = torch.sqrt(s_pwr / (snr * n_pwr + 1e-7))
    
    return clean_wf + (scale * noise_wf)

In [15]:
limit = len(df_all_files)
count_processed = 0

print(f"🚀 Memulai RespiScan Stable Pipeline pada {device}...")

for i, row in df_all_files.iterrows():
    try:
        f_name = row['filename']
        f_label = row['Label']
        f_path = row['full_path']
        
        # --- CHECKPOINT: Lewati jika sudah pernah diproses ---
        if is_already_processed(f_name, f_label, output_dir, "_clean") and \
           is_already_processed(f_name, f_label, output_dir, "_noisy"):
            continue

        # 1. Load & Standardize (CPU)
        wf, sr = torchaudio.load(f_path)
        wf = standardize_audio(wf, sr) # Gunakan fungsi standardize_audio yg kita buat tadi

        # 2. Proses Clean
        if not is_already_processed(f_name, f_label, output_dir, "_clean"):
            extract_features_stable(wf, f_name, f_label, "_clean", output_dir, device)
        
        # 3. Proses Noisy
        if not is_already_processed(f_name, f_label, output_dir, "_noisy"):
            noise_wf = get_random_noise_segment(noise_dirs) # Fungsi noise tadi
            noisy_wf = mix_noise(wf, noise_wf, snr_db=random.randint(5, 15))
            extract_features_stable(noisy_wf, f_name, f_label, "_noisy", output_dir, device)

        # 4. Maintenance berkala
        count_processed += 1
        if i % 10 == 0:
            print(f"✅ Selesai: {i+1}/{limit} | RAM terkelola", end='\r')
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()

    except Exception as e:
        print(f"\n⚠️ Error pada file {f_name}: {e}")
        continue

print(f"\n\n✨ Selesai! Berhasil memproses {count_processed} file baru.")


🚀 Memulai RespiScan Stable Pipeline pada cuda...


KeyboardInterrupt: 

In [2]:
# MFCC DAN SCALOGRAM MENGOLAH DATA BARU NTB
import os
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt
import random

# --- KONFIGURASI PATH LOKAL ---
PATH_AUDIO_NTB = r'C:\TBC SCREENING PENELITIAN\ntb_bulk_10k\TBscreen_Dataset\Passive_coughs\Audio_files'
PATH_OUTPUT_FINAL = r'C:\TBC SCREENING PENELITIAN\RespiScan_Train_Ready'

# Berapa AUDIO yang dibutuhkan untuk menghasilkan jumlah gambar yang pas (1 audio = 2 gambar)
GAP_AUDIO = {
    'TRAIN': 1676, 
    'VAL': 222,
    'TEST': 366
}

def add_noise(y, noise_factor=0.005):
    noise = np.random.randn(len(y))
    return y + noise_factor * noise

def save_as_image(data, sr, out_path, mode='mfcc'):
    plt.figure(figsize=(10, 4))
    if mode == 'mfcc':
        feat = librosa.feature.mfcc(y=data, sr=sr, n_mfcc=40)
    else: # Scalogram pendekatan Mel-Spec
        feat = librosa.feature.melspectrogram(y=data, sr=sr, n_mels=128)
        feat = librosa.power_to_db(feat, ref=np.max)
        
    librosa.display.specshow(feat, sr=sr)
    plt.axis('off')
    plt.savefig(out_path, bbox_inches='tight', pad_inches=0)
    plt.close()

# --- PROSES EKSEKUSI ---
all_wavs = [f for f in os.listdir(PATH_AUDIO_NTB) if f.endswith('.wav')]
random.shuffle(all_wavs) # Acak agar ID subjek tersebar

current_idx = 0
for split, needed_audio in GAP_AUDIO.items():
    dest_dir = os.path.join(PATH_OUTPUT_FINAL, split, 'Healthy')
    if not os.path.exists(dest_dir):
        os.makedirs(dest_dir)
    
    print(f"🚀 Memproses {needed_audio} audio baru untuk set {split}...")
    
    done_audio = 0
    while done_audio < needed_audio and current_idx < len(all_wavs):
        file_name = all_wavs[current_idx]
        file_path = os.path.join(PATH_AUDIO_NTB, file_name)
        file_id = file_name.replace('.wav', '')
        
        try:
            y, sr = librosa.load(file_path, sr=22050)
            y_noisy = add_noise(y) # Tambah kebisingan

            # Simpan MFCC
            save_as_image(y_noisy, sr, os.path.join(dest_dir, f"MFCC_{file_id}.png"), 'mfcc')
            # Simpan Scalogram
            save_as_image(y_noisy, sr, os.path.join(dest_dir, f"SCAL_{file_id}.png"), 'scal')
            
            done_audio += 1
        except Exception as e:
            print(f"Error pada {file_name}: {e}")
        
        current_idx += 1
        if done_audio % 50 == 0:
            print(f"   ✅ {done_audio}/{needed_audio} audio selesai diproses...")

print("\n✨ MANTAP! Folder Healthy sekarang sudah seimbang dengan TB.")

🚀 Memproses 1676 audio baru untuk set TRAIN...


c:\Users\Gibran\miniconda3\envs\dsscience\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


   ✅ 50/1676 audio selesai diproses...
   ✅ 100/1676 audio selesai diproses...
   ✅ 150/1676 audio selesai diproses...
   ✅ 200/1676 audio selesai diproses...
   ✅ 250/1676 audio selesai diproses...
   ✅ 300/1676 audio selesai diproses...
   ✅ 350/1676 audio selesai diproses...
   ✅ 400/1676 audio selesai diproses...
   ✅ 450/1676 audio selesai diproses...
🚀 Memproses 222 audio baru untuk set VAL...
🚀 Memproses 366 audio baru untuk set TEST...

✨ MANTAP! Folder Healthy sekarang sudah seimbang dengan TB.


In [6]:
import pandas as pd

# 1. Ekstrak PID dari nama file (Asumsi format: PID_XXXX_...)
# Kita ambil bagian pertama sebelum underscore
df_all_files['PID'] = df_all_files['filename'].apply(lambda x: x.split('_')[0] + "_" + x.split('_')[1])

# 2. Hitung jumlah subjek unik total
total_subjects = df_all_files['PID'].nunique()

# 3. Hitung distribusi subjek per Label (TB vs Healthy)
subject_dist = df_all_files.groupby('Label')['PID'].nunique()

print("📊 Audit Variabilitas Subjek RespiScan:")
print(f"Total File: {len(df_all_files)}")
print(f"Total Subjek Unik: {total_subjects}")
print("-" * 30)
print(f"Subjek di Kelas Healthy: {subject_dist.get(0, 0)}")
print(f"Subjek di Kelas TB: {subject_dist.get(1, 0)}")

# 4. Cek apakah ada satu subjek yang batuknya terlalu banyak (Dominasi data)
top_subjects = df_all_files['PID'].value_counts().head(5)
print("\n🔝 5 Subjek dengan kontribusi file terbanyak:")
print(top_subjects)

📊 Audit Variabilitas Subjek RespiScan:
Total File: 15814
Total Subjek Unik: 1952
------------------------------
Subjek di Kelas Healthy: 380
Subjek di Kelas TB: 1572

🔝 5 Subjek dengan kontribusi file terbanyak:
PID
PID_117A    1475
PID_144A     915
PID_138A     601
PID_22C      485
PID_182A     482
Name: count, dtype: int64


C:\Users\Gibran\AppData\Local\Temp\ipykernel_15808\119293682.py:17: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Subjek di Kelas Healthy: {subject_dist.get(0, 0)}")
C:\Users\Gibran\AppData\Local\Temp\ipykernel_15808\119293682.py:18: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"Subjek di Kelas TB: {subject_dist.get(1, 0)}")


In [7]:
import os
import shutil
import pandas as pd
from sklearn.model_selection import train_test_split
from tqdm import tqdm

# --- 1. KONFIGURASI ---
source_dir = r'C:\TBC SCREENING PENELITIAN\RespiScan_Final_Dataset'
target_base = r'C:\TBC SCREENING PENELITIAN\RespiScan_Train_Ready'
MAX_FILES_PER_PID = 50  

# --- 2. PERSIAPAN DATA ---
print("🔄 Menyiapkan daftar subjek...")
# Ekstraksi PID (Contoh: PID_108A0)
df_all_files['PID'] = df_all_files['filename'].apply(lambda x: str(x).split('_')[0] + "_" + str(x).split('_')[1])

# Pisahkan PID (Gunakan label NTB dan TB sesuai audit kamu)
healthy_pids = df_all_files[df_all_files['Label'] == 'NTB']['PID'].unique()
tb_pids = df_all_files[df_all_files['Label'] == 'TB']['PID'].unique()

def split_subjects(pids):
    if len(pids) == 0: return [], [], []
    train, temp = train_test_split(pids, test_size=0.3, random_state=42)
    val, test = train_test_split(temp, test_size=0.5, random_state=42)
    return train, val, test

h_train, h_val, h_test = split_subjects(healthy_pids)
tb_train, tb_val, tb_test = split_subjects(tb_pids)

split_map = {
    'train': {'Healthy': h_train, 'TB': tb_train},
    'val': {'Healthy': h_val, 'TB': tb_val},
    'test': {'Healthy': h_test, 'TB': tb_test}
}

# --- 3. EKSEKUSI PEMINDAHAN ---
print("🚚 Memindahkan file ke folder baru (RespiScan Ready)...")
total_moved = 0
not_found_count = 0

for set_name, labels in split_map.items():
    for label_name, pids in labels.items():
        # Folder tujuan
        dest_path = os.path.join(target_base, set_name, label_name)
        os.makedirs(dest_path, exist_ok=True)
        
        # Folder asal (NTB untuk Healthy, TB untuk TB)
        folder_asal = 'NTB' if label_name == 'Healthy' else 'TB'

        for pid in tqdm(pids, desc=f"Moving {set_name}/{label_name}"):
            pid_data = df_all_files[df_all_files['PID'] == pid]
            selected = pid_data.sample(min(len(pid_data), MAX_FILES_PER_PID), random_state=42)
            
            for _, row in selected.iterrows():
                f_name = row['filename'] # Ini berisi 'PID_XXXX_yeti.wav'
                
                for suffix in ['_clean', '_noisy']:
                    # 1. Path Scalogram (.png ada di folder label)
                    img_file = f"{f_name}{suffix}.png"
                    src_img = os.path.join(source_dir, folder_asal, img_file)
                    
                    # 2. Path MFCC (.pt ada di root source_dir)
                    mfcc_file = f"{f_name}{suffix}_mfcc.pt"
                    src_mfcc = os.path.join(source_dir, mfcc_file)
                    
                    # Eksekusi Copy
                    if os.path.exists(src_img):
                        shutil.copy(src_img, os.path.join(dest_path, img_file))
                        total_moved += 1
                        # Copy MFCC jika ada
                        if os.path.exists(src_mfcc):
                            shutil.copy(src_mfcc, os.path.join(dest_path, mfcc_file))
                    else:
                        not_found_count += 1

print(f"\n✨ BERHASIL!")
print(f"✅ Total file dipindahkan: {total_moved}")
print(f"⚠️ File tidak ditemukan: {not_found_count} (Cek apakah sudah diekstrak semua)")

🔄 Menyiapkan daftar subjek...
🚚 Memindahkan file ke folder baru (RespiScan Ready)...


Moving test/TB: 100%|██████████| 236/236 [00:13<00:00, 18.07it/s]


✨ BERHASIL!
✅ Total file dipindahkan: 7158
⚠️ File tidak ditemukan: 6296 (Cek apakah sudah diekstrak semua)


In [ ]:
#AUGMENTASI DATA NTB UNTUK MENYEIMBANGKAN DATASET
import os
import librosa
import librosa.display
import numpy as np
import matplotlib.pyplot as plt

# --- KONFIGURASI PATH LOKAL EL ---
PATH_AUDIO_NTB = r'C:\TBC SCREENING PENELITIAN\ntb_bulk_10k\TBscreen_Dataset\Passive_coughs\Audio_files'
PATH_OUTPUT = r'C:\TBC SCREENING PENELITIAN\RespiScan_Train_Ready'

def save_img(y, sr, folder, file_id, variant, type_img):
    plt.figure(figsize=(10, 4))
    if type_img == 'MFCC':
        feat = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=40)
    else:
        feat = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr), ref=np.max)
    librosa.display.specshow(feat, sr=sr)
    plt.axis('off')
    # Nama file dibedakan agar tidak menimpa yang lama
    plt.savefig(os.path.join(folder, f"{type_img}_{variant}_{file_id}.png"), bbox_inches='tight', pad_inches=0)
    plt.close()

# --- PROSES MULTI-AUGMENTASI ---
all_wavs = [f for f in os.listdir(PATH_AUDIO_NTB) if f.endswith('.wav')]

# Kita bagi 450 audio ini ke folder yang masih "haus" data
splits = {
    'TRAIN': all_wavs[:300], # 300 audio x 6 variasi baru = 1800 gambar tambahan
    'VAL': all_wavs[300:360], # 60 audio x 6 variasi baru = 360 gambar tambahan
    'TEST': all_wavs[360:]    # 90 audio x 6 variasi baru = 540 gambar tambahan
}

for split, wav_list in splits.items():
    dest_dir = os.path.join(PATH_OUTPUT, split, 'Healthy')
    print(f"🚀 Menambahkan variasi baru ke set {split}...")

    for wav in wav_list:
        file_id = wav.replace('.wav', '')
        y, sr = librosa.load(os.path.join(PATH_AUDIO_NTB, wav), sr=22050)

        # Variasi A: Pitch Up (Suara Naik)
        y_up = librosa.effects.pitch_shift(y, sr=sr, n_steps=2)
        save_img(y_up, sr, dest_dir, file_id, "PITCHUP", "MFCC")
        save_img(y_up, sr, dest_dir, file_id, "PITCHUP", "SCAL")

        # Variasi B: Pitch Down (Suara Turun/Ngebass)
        y_down = librosa.effects.pitch_shift(y, sr=sr, n_steps=-2)
        save_img(y_down, sr, dest_dir, file_id, "PITCHDOWN", "MFCC")
        save_img(y_down, sr, dest_dir, file_id, "PITCHDOWN", "SCAL")

        # Variasi C: Fast (Tempo dipercepat)
        y_fast = librosa.effects.time_stretch(y, rate=1.1)
        save_img(y_fast, sr, dest_dir, file_id, "FAST", "MFCC")
        save_img(y_fast, sr, dest_dir, file_id, "FAST", "SCAL")

print("\n✨ BERES! Sekarang cek folder Healthy kamu, pasti jumlahnya sudah melonjak drastis.")

🚀 Menambahkan variasi baru ke set TRAIN...
🚀 Menambahkan variasi baru ke set VAL...
🚀 Menambahkan variasi baru ke set TEST...

✨ BERES! Sekarang cek folder Healthy kamu, pasti jumlahnya sudah melonjak drastis.


In [8]:
import os

def final_audit(target_path):
    for set_name in ['train', 'val', 'test']:
        print(f"📁 Set: {set_name.upper()}")
        for label in ['Healthy', 'TB']:
            path = os.path.join(target_path, set_name, label)
            if os.path.exists(path):
                # Hitung hanya file .png (Scalogram)
                n_files = len([f for f in os.listdir(path) if f.endswith('.png')])
                print(f"   - {label}: {n_files} gambar")
        print("-" * 20)

final_audit(target_base)


📁 Set: TRAIN
   - Healthy: 3306 gambar
   - TB: 3954 gambar
--------------------
📁 Set: VAL
   - Healthy: 598 gambar
   - TB: 682 gambar
--------------------
📁 Set: TEST
   - Healthy: 784 gambar
   - TB: 868 gambar
--------------------


In [9]:
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

def plot_samples(base_path, class_name, num_samples=3):
    folder = os.path.join(base_path, 'TRAIN', class_name)
    files = random.sample(os.listdir(folder), num_samples)
    
    fig, axes = plt.subplots(1, num_samples, figsize=(15, 5))
    fig.suptitle(f'Sample Gambar Kelas: {class_name}', fontsize=16)
    
    for i, file in enumerate(files):
        img = Image.open(os.path.join(folder, file))
        axes[i].imshow(img)
        axes[i].set_title(file.split('_')[0]) # Menampilkan jenis (MFCC/SCAL)
        axes[i].axis('off')
    plt.show()

# Jalankan untuk kedua kelas
PATH_DATA = r'C:\TBC SCREENING PENELITIAN\RespiScan_Train_Ready'
plot_samples(PATH_DATA, 'Healthy')
plot_samples(PATH_DATA, 'TB')

C:\Users\Gibran\AppData\Local\Temp\ipykernel_15808\4111068592.py:18: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


UnidentifiedImageError: cannot identify image file 'C:\\TBC SCREENING PENELITIAN\\RespiScan_Train_Ready\\TRAIN\\TB\\PID_212A_14_pixel.wav_noisy_mfcc.pt'

In [10]:
#MEMBANGUN MODEL
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from PIL import Image
import random
import numpy as np

# --- KONFIGURASI PATH ---
# Gunakan folder yang sudah berisi data seimbang hasil augmentasi kamu
DATA_DIR = r'C:\TBC SCREENING PENELITIAN\RespiScan_Train_Ready'
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"✅ Perangkat aktif: {DEVICE}")
print(f"📂 Dataset utama: {DATA_DIR}")


ModuleNotFoundError: No module named 'torchvision'